# 设计模式实战

> 通过 Agent 项目学习常见设计模式的应用

In [1]:
# === 环境设置 ===
import sys
import os
from pathlib import Path

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

# 加载 .env
env_file = Path(project_path) / ".env" if project_path else None
if env_file and env_file.exists():
    with open(env_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

print(f"项目路径: {project_path}")

# 验证模块导入
try:
    from backend.llm import LLMFactory
    from backend.tools.base import BaseToolRegistry
    from backend.agents import BaseAgent, AgentState
    print("✓ 所有模块导入成功")
except ImportError as e:
    print(f"✗ 模块导入失败: {e}")

项目路径: c:\Users\zying\Desktop\pro_agent\agent-learning-project
✓ 所有模块导入成功


## 项目中的设计模式概览

| 设计模式 | 位置 | 作用 |
|----------|------|------|
| **工厂模式** | `LLMFactory`, `EmbedderFactory` | 创建不同提供商的对象 |
| **单例模式** | `ToolRegistry` | 全局唯一的工具注册中心 |
| **装饰器模式** | `@register_tool` | 零侵入的工具注册 |
| **模板方法模式** | `BaseAgent`, `BaseLLM`, `BaseTool` | 定义算法骨架 |
| **管道模式** | `RAGPipeline` | 分阶段处理数据流 |
| **策略模式** | 各 LLM 实现类 | 可互换的算法族 |
| **状态模式** | `AgentState` | 根据状态改变行为 |

---

## 1. 工厂模式 (Factory Pattern)

### 定义

定义一个创建对象的接口，让子类决定实例化哪一个类。

### 为什么需要工厂模式？

```python
# 没有工厂模式 - 客户端代码依赖具体类
if provider == "openai":
    llm = OpenAILLM(api_key=key)
elif provider == "anthropic":
    llm = AnthropicLLM(api_key=key)
elif provider == "deepseek":
    llm = DeepSeekLLM(api_key=key)
# 每次新增提供商都要修改客户端代码！

# 有工厂模式 - 客户端代码不依赖具体类
llm = LLMFactory.create(provider)  # 一行搞定
```

### 项目实现：LLMFactory

In [2]:
from backend.llm.factory import LLMFactory

# 查看工厂类内部结构
print("=== LLMFactory 工厂模式 ===\n")

print("已注册的提供商:")
for name, cls in LLMFactory._providers.items():
    print(f"  '{name}' → {cls.__name__}")

=== LLMFactory 工厂模式 ===

已注册的提供商:
  'openai' → OpenAILLM
  'anthropic' → AnthropicLLM
  'ollama' → OllamaLLM
  'glm' → GLMLLM
  'deepseek' → DeepSeekLLM


### 工厂模式结构图

```
┌─────────────────────────────────────────────────────┐
│                   客户端代码                        │
│  LLMFactory.create("deepseek")                     │
└────────────────────┬────────────────────────────────┘
                     ↓
┌─────────────────────────────────────────────────────┐
│                  LLMFactory (工厂)                  │
│  _providers = {                                     │
│    "openai": OpenAILLM,                           │
│    "anthropic": AnthropicLLM,                     │
│    "deepseek": DeepSeekLLM,                       │
│  }                                                  │
│  create(provider) → 实例                            │
└────────────────────┬────────────────────────────────┘
                     ↓
       ┌─────────────┼─────────────┐
       ↓             ↓             ↓
  ┌─────────┐  ┌─────────┐  ┌─────────┐
  │OpenAILLM│  │Anthropic│  │DeepSeek │
  └────┬────┘  └────┬────┘  └────┬────┘
       └─────────────┴─────────────┘
                     ↓
            ┌────────────────┐
            │   BaseLLM      │ (抽象基类)
            │  + chat()      │
            │  + embed()     │
            └────────────────┘
```

### 练习：使用工厂创建 LLM 实例

In [3]:
import os
from pathlib import Path

# 获取 API key
def get_deepseek_key():
    project_path = Path(os.getcwd())
    for path in [project_path] + list(project_path.parents):
        env_file = path / ".env"
        if env_file.exists():
            with open(env_file, encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line.startswith("DEEPSEEK_API_KEY="):
                        return line.split("=", 1)[1].strip()
    return ""

# 使用工厂创建不同提供商的实例
api_key = get_deepseek_key()

print("=== 使用工厂模式创建 LLM 实例 ===\n")

# DeepSeek
deepseek_llm = LLMFactory.create("deepseek", api_key=api_key)
print(f"DeepSeek LLM:")
print(f"  类型: {type(deepseek_llm).__name__}")
print(f"  模型: {deepseek_llm.model}")
print(f"  支持 tools: {deepseek_llm.supports_tools()}")

=== 使用工厂模式创建 LLM 实例 ===

DeepSeek LLM:
  类型: DeepSeekLLM
  模型: deepseek-chat
  支持 tools: True


---

## 2. 单例模式 (Singleton Pattern)

### 定义

确保一个类只有一个实例，并提供全局访问点。

### 为什么需要单例模式？

```
工具注册中心应该只有一个！

如果有多个实例：
  - Agent A 注册的工具，Agent B 看不到
  - 造成不一致和混乱
```

### 项目实现：ToolRegistry

In [4]:
from backend.tools.registry import ToolRegistry

print("=== ToolRegistry 单例模式 ===\n")

# 创建多个实例
registry1 = ToolRegistry()
registry2 = ToolRegistry()

print(f"registry1 id: {id(registry1)}")
print(f"registry2 id: {id(registry2)}")
print(f"是否是同一个实例: {registry1 is registry2}")

print(f"\n已注册工具数量: {len(ToolRegistry.list_all())}")
print(f"工具列表: {ToolRegistry.list_all()}")

=== ToolRegistry 单例模式 ===

registry1 id: 2256943412432
registry2 id: 2256943412432
是否是同一个实例: True

已注册工具数量: 9
工具列表: ['search', 'ddg_search', 'wikipedia_search', 'calculator', 'advanced_calculator', 'read_file', 'write_file', 'list_files', 'create_directory']


### 单例模式实现原理

```python
class ToolRegistry:
    _instance = None  # 存储唯一实例
    
    def __new__(cls):
        if cls._instance is None:  # 只创建一次
            cls._instance = super().__new__(cls)
        return cls._instance  # 始终返回同一个
```

```
第一次调用 ToolRegistry():
  _instance 是 None → 创建新实例 → 返回

第二次调用 ToolRegistry():
  _instance 已存在 → 直接返回已存在的实例
```

---

## 3. 装饰器模式 (Decorator Pattern)

### 定义

动态地给一个对象添加额外的职责。

### 项目实现：@register_tool

In [5]:
from backend.tools import register_tool, BaseTool
from typing import Optional

print("=== 装饰器模式：@register_tool ===\n")

# 先清空，演示注册过程
ToolRegistry.clear()
print(f"清空后工具数量: {len(ToolRegistry.list_all())}")

# 定义一个自定义工具
@register_tool
class MyDemoTool(BaseTool):
    name = "demo_tool"
    description = "这是一个演示工具"
    
    parameters = []  # 无参数
    
    async def execute(self, **kwargs):
        return {"result": "Hello from demo tool!"}

print(f"\n使用 @register_tool 装饰后:")
print(f"工具数量: {len(ToolRegistry.list_all())}")
print(f"工具列表: {ToolRegistry.list_all()}")

# 获取并使用工具
tool = ToolRegistry.get("demo_tool")
print(f"\n获取工具: {tool.name}")
print(f"工具描述: {tool.description}")

=== 装饰器模式：@register_tool ===

清空后工具数量: 0

使用 @register_tool 装饰后:
工具数量: 1
工具列表: ['demo_tool']

获取工具: demo_tool
工具描述: 这是一个演示工具


### 装饰器工作原理

```python
# 这些代码是等价的：

# 方式 1: 装饰器语法
@register_tool
class MyTool(BaseTool):
    pass

# 方式 2: 直接调用
class MyTool(BaseTool):
    pass
MyTool = register_tool(MyTool)

# 装饰器做了什么：
def register_tool(tool_class):
    tool_instance = tool_class()      # 创建实例
    ToolRegistry.register(tool_instance)  # 注册
    return tool_class                 # 返回原类
```

---

## 4. 模板方法模式 (Template Method Pattern)

### 定义

定义算法的骨架，将一些步骤延迟到子类实现。

### 项目实现：BaseLLM 抽象基类

In [6]:
from abc import ABC, abstractmethod

print("=== 模板方法模式：BaseLLM ===\n")

# 查看 BaseLLM 的抽象方法
from backend.llm.base import BaseLLM

import inspect
print("BaseLLM 定义的抽象方法（子类必须实现）:")
for name, method in inspect.getmembers(BaseLLM, predicate=inspect.isfunction):
    if getattr(method, '__isabstractmethod__', False):
        sig = inspect.signature(method)
        print(f"  - {name}{sig}")

=== 模板方法模式：BaseLLM ===

BaseLLM 定义的抽象方法（子类必须实现）:
  - chat(self, messages: List[backend.llm.models.Message], tools: List[backend.llm.models.ToolDefinition] | None = None, temperature: float = 0.7, max_tokens: int | None = None, **kwargs) -> backend.llm.models.ChatResponse
  - embed(self, texts: List[str], **kwargs) -> backend.llm.models.EmbeddingResponse


### 模板方法模式结构

```
┌─────────────────────────────────────┐
│         BaseLLM (抽象类)            │
│  ┌─────────────────────────────┐    │
│  │ + chat(): ChatResponse      │    │
│  │   (抽象方法，子类必须实现)    │    │
│  └─────────────────────────────┘    │
│  ┌─────────────────────────────┐    │
│  │ + embed(): EmbeddingResponse│    │
│  │   (抽象方法，子类必须实现)    │    │
│  └─────────────────────────────┘    │
│  ┌─────────────────────────────┐    │
│  │ + supports_tools(): bool    │    │
│  │   (具体方法，已有默认实现)    │    │
│  └─────────────────────────────┘    │
└──────────────┬──────────────────────┘
               ↓ 继承
       ┌───────┴────────┐
       ↓                ↓
┌────────────┐  ┌────────────┐
│ OpenAILLM  │  │ DeepSeek   │
│ + chat()   │  │ + chat()   │
│  (实现细节)│  │  (实现细节)│
└────────────┘  └────────────┘
```

### 模板方法模式的好处

```python
# 1. 统一接口 - 所有 LLM 用法一致
async def ask_llm(llm: BaseLLM, question: str):
    response = await llm.chat([Message(role="user", content=question)])
    return response.content

# 可以传入任何 BaseLLM 子类
await ask_llm(OpenAILLM(...), "你好")
await ask_llm(DeepSeekLLM(...), "你好")
await ask_llm(AnthropicLLM(...), "你好")

# 2. 代码复用 - 公共逻辑在基类
# 3. 扩展性 - 新增 LLM 只需实现抽象方法
```

---

## 5. 管道模式 (Pipeline Pattern)

### 定义

将复杂处理流程分解为多个阶段，每个阶段独立且可替换。

### 项目实现：RAGPipeline

In [7]:
from backend.rag.pipeline import RAGPipeline

print("=== 管道模式：RAGPipeline ===\n")

# 查看 Pipeline 的结构
print("RAGPipeline 的组件:")
pipeline = RAGPipeline()
print(f"  - embedder: {type(pipeline.embedder).__name__}")
print(f"  - vector_store: {type(pipeline.vector_store).__name__}")
print(f"  - retriever: {type(pipeline.retriever).__name__}")

=== 管道模式：RAGPipeline ===

RAGPipeline 的组件:
  - embedder: OpenAIEmbedder
  - vector_store: VectorStore
  - retriever: RAGRetriever


### 管道模式数据流

```
输入文档
    ↓
┌─────────────────────────────────────────────┐
│              RAG Pipeline                    │
│  ┌─────────┐  ┌─────────┐  ┌─────────┐       │
│  │Embedder │→ │Vector   │→ │Retriever│       │
│  │向量化   │  │Store    │  │语义检索 │       │
│  └─────────┘  └─────────┘  └────┬────┘       │
│                              ↓              │
│                         ┌─────────┐          │
│                         │   LLM   │          │
│                         │生成答案 │          │
│                         └────┬────┘          │
└──────────────────────────────┼───────────────┘
                               ↓
                          答案 + 来源
```

### 管道模式的好处

| 好处 | 说明 |
|------|------|
| **解耦** | 每个阶段独立，互不影响 |
| **可测试** | 可以单独测试每个阶段 |
| **可替换** | 换一个 Embedder 不影响其他部分 |
| **可扩展** | 添加新阶段（如重排序）很容易 |

---

## 6. 策略模式 (Strategy Pattern)

### 定义

定义一系列算法，把它们封装起来，并使它们可以互换。

### 项目中的策略模式

In [8]:
print("=== 策略模式：不同 LLM 实现 ===\n")

# 各种 LLM 实现就是不同的策略
from backend.llm.openai_client import OpenAILLM
from backend.llm.deepseek_client import DeepSeekLLM
from backend.llm.anthropic_client import AnthropicLLM

print("不同的 LLM 策略:")
for cls in [OpenAILLM, DeepSeekLLM, AnthropicLLM]:
    print(f"  - {cls.__name__}: {cls.__doc__ or '无描述'}")

print("\n策略模式 - 它们都实现相同的接口 (BaseLLM):")
print("  但内部实现不同（调用不同 API）")

=== 策略模式：不同 LLM 实现 ===

不同的 LLM 策略:
  - OpenAILLM: 
OpenAI LLM客户端

支持模型：
- gpt-4o, gpt-4-turbo, gpt-4
- gpt-3.5-turbo
- o1-preview, o1-mini

支持：
- 聊天对话（带工具调用）
- 文本嵌入

  - DeepSeekLLM: 
DeepSeek LLM客户端

支持模型：
- deepseek-chat: 通用对话模型
- deepseek-coder: 代码模型

DeepSeek使用OpenAI兼容的API，文档: https://platform.deepseek.com/api-docs/

  - AnthropicLLM: 
Anthropic (Claude) LLM客户端

支持模型：
- claude-3-5-sonnet-20241022
- claude-3-opus-20240229
- claude-3-sonnet-20240229
- claude-3-haiku-20240307

注意：
- Anthropic目前不提供官方嵌入API
- 工具调用格式与OpenAI略有不同


策略模式 - 它们都实现相同的接口 (BaseLLM):
  但内部实现不同（调用不同 API）


### 策略模式结构

```
┌─────────────────────────────────────┐
│         客户端代码                  │
│  llm = LLMFactory.create(provider)  │
│  result = await llm.chat(messages)  │
└──────────────┬──────────────────────┘
               ↓ 使用
┌─────────────────────────────────────┐
│        BaseLLM (策略接口)          │
│  + chat(messages)                   │
└──────┬──────────────────────────────┘
       ↓
  ┌────┴────────┐
  ↓             ↓           ↓
┌────────┐ ┌────────┐ ┌────────┐
│Strategy1│ │Strategy2│ │Strategy3│
│OpenAI   │ │DeepSeek │ │Anthropic│
│API调用  │ │API调用  │ │API调用  │
└────────┘ └────────┘ └────────┘
```

---

## 7. 状态模式 (State Pattern)

### 定义

允许一个对象在其内部状态改变时改变其行为。

### 项目实现：AgentState

In [9]:
from backend.agents.base import AgentState

print("=== 状态模式：AgentState ===\n")

print("Agent 可能的状态:")
for state in AgentState:
    print(f"  - {state.value}: {state.name}")

print("\n状态转换:")
print("  IDLE → THINKING → ACTING → THINKING → ... → DONE")

=== 状态模式：AgentState ===

Agent 可能的状态:
  - idle: IDLE
  - thinking: THINKING
  - acting: ACTING
  - done: DONE
  - error: ERROR

状态转换:
  IDLE → THINKING → ACTING → THINKING → ... → DONE


### 状态转换图

```
    ┌─────────┐
    │  IDLE   │  空闲，等待请求
    └────┬────┘
         │ 收到请求
         ↓
    ┌─────────┐
    │THINKING │  调用 LLM 思考
    └────┬────┘
         │ 需要工具
         ↓
    ┌─────────┐
    │ ACTING  │  执行工具
    └────┬────┘
         │ 工具返回
         ↓
    ┌─────────┐
    │THINKING │  基于结果再思考
    └────┬────┘
         │ 完成任务
         ↓
    ┌─────────┐
    │  DONE   │  返回结果
    └─────────┘
```

---

## 设计模式总结对比

| 模式 | 核心思想 | 解决的问题 | 项目应用 |
|------|----------|------------|----------|
| **工厂** | 创建对象分离 | 客户端不依赖具体类 | LLMFactory |
| **单例** | 全局唯一实例 | 避免重复创建 | ToolRegistry |
| **装饰器** | 动态添加功能 | 不修改原代码 | @register_tool |
| **模板方法** | 算法骨架 | 统一接口，复用代码 | BaseLLM, BaseTool |
| **管道** | 分阶段处理 | 流程清晰，易于扩展 | RAGPipeline |
| **策略** | 算法可互换 | 运行时选择算法 | 不同 LLM 实现 |
| **状态** | 状态决定行为 | 复杂状态管理 | AgentState |

---

## 常见面试问题

**Q: 工厂模式和简单工厂有什么区别？**
- A: 简单工厂用一个静态方法和 if-else 创建对象；工厂模式定义工厂接口，每个具体工厂负责创建一种产品。本项目使用的是简单工厂变体。

**Q: 单例模式的线程安全如何保证？**
- A: 本项目的 `__new__` 实现不是线程安全的。线程安全版本可以用：
  1. 加锁 (`threading.Lock`)
  2. 使用模块级变量（Python 导入是线程安全的）

**Q: 装饰器和 @wraps 的作用？**
- A: `@wraps` 复制原函数的元数据（`__name__`, `__doc__`），保持被装饰函数的签名信息。

**Q: 抽象基类 (ABC) 和接口有什么区别？**
- A: Python 没有真正的 interface 关键字，ABC 通过 `@abstractmethod` 模拟接口。与纯接口相比，ABC 可以包含部分实现。

**Q: 策略模式和状态模式的区别？**
- A: 策略模式：算法可互换，客户端主动选择策略
  状态模式：状态决定行为，状态之间可以自动转换

**Q: 什么时候用管道模式？**
- A: 当数据需要经过多个处理阶段，且：
  - 各阶段相对独立
  - 需要灵活组合不同阶段
  - 需要单独测试每个阶段

**Q: 本项目为什么不用责任链模式？**
- A: 责任链适用于请求沿链传递直到被处理。本项目的 Agent 使用 ReAct 循环（思考-行动-观察），不是链式处理，所以不需要责任链。

**Q: 如何实现一个可扩展的插件系统？**
- A: 结合 装饰器 + 注册表 + 工厂模式：
  1. 定义插件接口 (ABC)
  2. 用 `@register_plugin` 装饰器注册
  3. PluginFactory 根据名称创建实例
  - 这正是本项目工具系统的实现方式！